# Intro to PEST, PstFrom and PESTPP-IES

_The model is a black box. Here is the contract that lets PEST talk to it._


Everything in this curriculum that adjusts parameters, runs ensembles or quantifies uncertainty rests on one idea: PEST never looks inside your model. It treats the model as a black box that reads numbers from text files and writes numbers to text files. If you can describe *where the numbers go in* and *where the numbers come out*, PEST can drive the model without knowing or caring that it is MODFLOW 6 coupled to PHREEQC.

This notebook is standalone background. It does three things:

1. lays out the **model-as-black-box contract** — "template" files, "instruction" files, and the "control" file that ties them together;
2. shows where `pyemu` and `PstFrom` sit in that stack (the tools that build the contract for you); and
3. explains how **PESTPP-IES**, the iterative ensemble smoother, uses that contract to history match a whole *ensemble* at once — in pictures and plain language.

A short, concrete demonstration on a toy text file makes the template/instruction mechanics tangible. It is pure Python: no model is run, nothing expensive happens.


This is part of the part0 background series. If the words "prior", "realisation", "ensemble" or "posterior" are new, read [`part0_03_uq_for_rtm`](../part0_03_uq_for_rtm/uq_for_rtm.ipynb) first — we use them here without re-deriving them. The companion [`part0_05_intro_to_dsi`](../part0_05_intro_to_dsi/intro_to_dsi.ipynb) picks up where this one leaves off, replacing the model in the loop with a cheap emulator. The machinery introduced here meets the DIZON model for real in [`part1_02_pstfrom_setup`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb).


### Admin

The only things we need here are `pyemu` (for its PEST file utilities) and the usual scientific-Python stack. No model files, no binaries, no parallel runs. The asserts below confirm we are using the vendored dependencies that ship with this repository rather than whatever happens to be installed on your machine.


In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
import pyemu
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10

We will do the toy demonstration in a throwaway folder created inside this notebook's own directory. (Every notebook in the series keeps its scratch files local to itself.)


In [ ]:
demo_d = "toy_interface"
if os.path.exists(demo_d):
    shutil.rmtree(demo_d)
os.makedirs(demo_d)

## The model-as-black-box contract

PEST (and PEST++) drive a model through an interface built entirely out of text files. The interface has exactly three kinds of file, and one loop:

![the forward run loop](./loop.png)

On each "forward run" PEST:

1. writes proposed parameter values into the model's input files, then
2. runs the model (whatever command that is), then
3. reads the simulated values it cares about back out of the model's output files.

Steps 1 and 3 are where the three file types live.


**Template files** (the `.tpl` files) handle step 1 — getting parameters *in*. A template file is a copy of a model input file with the numbers-to-be-adjusted blanked out and replaced by markers carrying parameter names. PEST reads the template, drops the current value of each named parameter into its marked slot, and writes a valid model input file. One template file per input file that contains adjustable parameters.

**Instruction files** (the `.ins` files) handle step 3 — getting results *out*. An instruction file is a little recipe that tells PEST how to navigate a model output file and pluck out the specific numbers that correspond to named "observations" (simulated equivalents of measured data, plus any forecasts we want to track). One instruction file per output file we need to read.

**The control file** (the `.pst` file) is the contract itself. It lists every parameter (name, initial value, bounds, whether it is adjustable), every observation (name, measured value, weight, group), which template file pairs with which model input file, which instruction file pairs with which output file, and the command PEST should run to execute the model. It is the single document that defines the whole inverse problem.


That is the entire contract. Note what is *not* in it: nothing about MODFLOW, nothing about PHREEQC, nothing about reaction networks. PEST sees parameters going into text files and observations coming out of text files. This is what lets the same tool drive a reactive transport model that takes ~6 min/run on a laptop, exactly as it would drive a one-second analytical model — the inverse problem is described in terms of the interface, not the physics.


## A toy interface, end to end

Let's make the contract concrete on a single text file. Imagine a trivial "model" whose input file sets two parameters and whose output file reports two simulated numbers. We will build the template and instruction files by hand, then use `pyemu`'s PEST file utilities to prove the round-trip works — all without running anything.


First, a stand-in for a model **input file**. Real models rarely use a layout this friendly, but the mechanics are identical regardless of formatting:


In [ ]:
input_txt = (
    "# reaction inputs\n"
    "pyrite_m0   1.5000\n"
    "oxid_rate   0.0030\n"
)
with open(os.path.join(demo_d, "model.in"), "w") as f:
    f.write(input_txt)
print(input_txt)

Now the **template file**. We copy the input file but replace each value we want PEST to adjust with a *marked* parameter name. The header line `ptf ~` declares this a PEST template file (`ptf` = "PEST template file") and announces that `~` is the marker delimiter. Anything between a pair of `~` characters is a parameter name; the *width* of the marked field tells PEST how many characters it may use when it writes the number back in.


In [ ]:
tpl_txt = (
    "ptf ~\n"
    "# reaction inputs\n"
    "pyrite_m0   ~  pyrite_m0  ~\n"
    "oxid_rate   ~  oxid_rate  ~\n"
)
tpl_file = os.path.join(demo_d, "model.in.tpl")
with open(tpl_file, "w") as f:
    f.write(tpl_txt)
print(tpl_txt)

`pyemu` can parse a template file and report the parameter names it found — handy as a sanity check, and exactly what the control-file builder does under the hood:


In [ ]:
par_names = pyemu.pst_utils.parse_tpl_file(tpl_file)
par_names

Now the half that matters most: writing a model input file *from* the template, given a set of parameter values. This is what PEST does at the start of every single forward run. We hand it a `dict` (or `pandas.Series`) of parameter values and it fills the marked slots:


In [ ]:
parvals = {"pyrite_m0": 2.25, "oxid_rate": 0.0071}
pyemu.pst_utils.write_to_template(parvals, tpl_file,
                                  os.path.join(demo_d, "model.in"))
print(open(os.path.join(demo_d, "model.in")).read())

There it is — the input file now carries the values PEST proposed, written in scientific notation sized to fit the marked fields. The model would read this file none the wiser. That is the whole of step 1.


Now step 3, the output side. Pretend the model just ran and wrote this **output file**, reporting a simulated peak SO4 and a simulated pH:


In [ ]:
output_txt = (
    "simulation results\n"
    "peak_so4   241.7   mg/L\n"
    "end_ph       6.83\n"
)
out_file = os.path.join(demo_d, "model.out")
with open(out_file, "w") as f:
    f.write(output_txt)
print(output_txt)

The **instruction file** tells PEST how to read it. The header `pif ~` declares a PEST instruction file (`pif` = "PEST instruction file") and again sets `~` as a marker. Each line is a navigation recipe: `l1` means "advance one line", a `~text~` marker means "scan forward to this literal string", and `!name!` means "read the next whitespace-delimited number and store it as observation `name`". The recipe below skips the header line, then on each of the next two lines anchors on the row label and grabs the number after it:


In [ ]:
ins_txt = (
    "pif ~\n"
    "l1\n"
    "~peak_so4~ !peak_so4!\n"
    "~end_ph~ !end_ph!\n"
)
ins_file = os.path.join(demo_d, "model.out.ins")
with open(ins_file, "w") as f:
    f.write(ins_txt)
print(ins_txt)

`pyemu` can apply the instruction file to the output file and hand back the observation values it extracted — this is precisely what PEST does to harvest results at the end of every forward run:


In [ ]:
obs = pyemu.pst_utils.try_process_output_file(ins_file, out_file)
obs

And that is the complete contract, exercised end to end: values written *in* through a template, results read *out* through an instruction file. A control file would simply record the `model.in.tpl` -> `model.in` and `model.out.ins` -> `model.out` pairings, the parameter bounds, the observation weights, and the command to run the model in between. PEST would then repeat the in-run-out loop hundreds or thousands of times, varying the parameters each time.


## Where `pyemu` and `PstFrom` sit

Hand-writing one template file for a two-line input file is fine. Hand-writing template and instruction files for a real model — hundreds of array files, time-series outputs at many monitoring sites and depths, thousands of parameters — is not. This is the job `pyemu` and, in particular, its `PstFrom` helper exist to do.

`pyemu` is a Python package for building and manipulating PEST datasets: it can read and write control files, parameter and observation data, ensembles, covariance matrices, and the tpl/ins files themselves (the utilities we just used by hand are the same ones it calls internally). `PstFrom` is the high-level assembler. You point it at a folder of model input files and tell it, in Python, *these arrays are hydraulic conductivity, parameterise them as pilot points with this geostatistical structure*, *these output columns are my SO4 observations* — and it generates the template files, the instruction files and a consistent control file for you, correlation structure and all.


So the stack, bottom to top:

- **the model** — reads input text files, writes output text files;
- **the tpl / ins / pst contract** — the text-file interface PEST drives the model through;
- **`pyemu` / `PstFrom`** — Python that *builds* that contract (and reads the results back for analysis);
- **PESTPP-IES** (and its siblings) — the engine that *uses* the contract to history match and quantify uncertainty.

We build the DIZON contract with `PstFrom` in [`part1_02_pstfrom_setup`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb), where the layered prior (flow, transport and reaction parameters) is wired up file by file. For now the point is only that the contract above is what `PstFrom` produces — there is no magic underneath, just a lot of carefully written template and instruction files.


## How PESTPP-IES works

PESTPP-IES is the engine we lean on most in this curriculum. "IES" stands for **iterative ensemble smoother**, and it embodies a deliberate break from the old "calibrate first, do uncertainty analysis later" habit. It does not search for a single "best" parameter set that "calibrates" the model. Instead it carries a whole *ensemble* of parameter sets and nudges all of them, together, toward the measured data — ending with a set of models that all fit history acceptably and that, run forward, give a *distribution* of the forecast.

_Why an ensemble instead of one best fit?_ Because the forecast is uncertain, and a single calibrated model hides that uncertainty behind one confident-looking number. The whole reason we are here is to answer a decision question with a probability attached — and you cannot get a probability out of a single model run.


### The algorithm in pictures

![the IES update loop](./ies_loop.png)

PESTPP-IES starts from an ensemble of parameter "realisations" sampled from the prior — each realisation is one plausible set of parameter values, and the spread of the ensemble encodes our prior uncertainty. It runs the model once for every realisation, giving a *prior* ensemble of simulated outputs. Then it iterates:

1. compare each realisation's simulated outputs against the measured data;
2. estimate, from the ensemble itself, how parameters and outputs co-vary — i.e. which parameter moves would reduce the misfit;
3. adjust every realisation a step in that direction;
4. re-run the model for the whole ensemble and repeat, for a handful of iterations.

The result is a *posterior* ensemble: parameter sets that all reproduce the measured history to a comparable degree. Run those forward over the forecast period and you have a posterior *forecast* distribution — uncertainty analysis as a by-product of the history match.


### The trick that makes it cheap

Step 2 is the clever part. Classical derivative-based methods (PESTPP-GLM, PEST_HP) compute how each output responds to each parameter by perturbing parameters one at a time — so the number of model runs per iteration grows with the *number of adjustable parameters*. With thousands of parameters that is ruinous, and for a ~6 min/run reactive transport model it is a non-starter.

PESTPP-IES never perturbs parameters individually. It reads the parameter-to-output relationships straight out of the *cross-covariances within the ensemble* — the realisations already sample many parameter combinations at once, so the relationships fall out of runs you were going to do anyway. The consequence is the headline feature:

> the number of model runs per iteration depends on the **ensemble size**, not on the number of adjustable parameters.

A few hundred runs can history match a model with thousands of parameters. That is why the method is feasible at all when each run is expensive.


### Observation noise carried through

There is one more piece. PESTPP-IES does not fit every realisation to the *same* measured values. It can pair each parameter realisation with its own realisation of the measured data *plus noise*, drawn from the noise statistics we assign each observation. Each realisation therefore chases a slightly different target, and the uncertainty contributed by measurement noise propagates straight into the posterior parameter and forecast spread. (This is why PESTPP-IES reports two objective-function summaries: "measured" includes the noise realisations, "actual" does not.)

Choosing those noise statistics honestly — proportional with a floor for concentrations, absolute for pH and temperature — is itself a judgment call, and we make it deliberately in [`part1_03_obs_weights_and_truth`](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb).


### A good fit is not a good forecast

One caution to carry forward, because it is the most important lesson the ensemble methods teach. With an imperfect model — and every model is imperfect — fitting the measured history *better* does not guarantee forecasting *better*. Pushing the ensemble to reproduce history tightly can quietly bias forecasts that depend on parameter combinations the data never constrained, and can shrink the posterior so far that it no longer brackets the true answer. Sometimes the more robust choice is to under-fit on purpose, or even to forgo history matching and report the prior.

Whether history matching helps *this* forecast — peak SO4 at the supply well, whose distribution sets the treatment capacity the operator must design for — is not assumed in this curriculum; it is something we check. The prior-versus-posterior forecast comparison in part1 is where that question gets answered honestly, rather than taken on faith.


### Tidy up

The toy interface has served its purpose. Remove the scratch folder:


In [ ]:
shutil.rmtree(demo_d)

## Where this goes next

You now have the whole stack in view: a model that speaks text files, the tpl/ins/pst contract that lets PEST drive it, `PstFrom` that builds the contract, and PESTPP-IES that uses it to history match an ensemble and hand back a forecast distribution.

In part1 this machinery meets the DIZON reactive transport model in earnest. [`part1_02_pstfrom_setup`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb) builds the control file and the layered prior; [`part1_04_prior_mc`](../part1_04_prior_mc/dizon_prior_mc.ipynb) runs the prior ensemble and asks whether we even need to history match. The catch — and the reason this series exists — is that DIZON costs ~6 min/run, so an honest PESTPP-IES history match of a few hundred realisations over several iterations runs to days of compute. [`part0_05_intro_to_dsi`](../part0_05_intro_to_dsi/intro_to_dsi.ipynb) introduces the way out: replace the expensive model in the loop with a cheap emulator trained on runs we have already paid for — keeping the exact same contract, at a tiny fraction of the cost.
